# `share_dispersion` Recalibration #2 — `late_access` / `early_access` (2050)

**Why:** after extending rule (i) (`GENSET_DIESEL.f_max` reopened in every 2050 scenario --
`2050/technologies.ipynb`), `late_access`/`early_access` re-solved with `PV_UTILITY` collapsing
(C4: ~160 -> ~75 MW) and some `GENSET_DIESEL` now built -- both change
`offre_locale_hors_TECH_HS`, so the `share_dispersion` calibrated on the *previous* solve no
longer bites correctly: check (a) drifted to 122-123% of target on C3 in both scenarios (>10%).
`early_access_brazil` is excluded -- its solve was essentially unchanged by the rule (i) extension
(`GENSET_DIESEL` still never gets built there, cheap import dominates), and its check (a) stayed
within 1% of target.

Same formula/denominator as every prior pass, targets unchanged. **One pass only**, per
instruction.


In [1]:
import pandas as pd
import json

ROOT = "../../../EnergyScope_BO_nord_amazonia"
CS_ROOT_2050 = f"{ROOT}/case_studies/C1_C2_C3_C4_C5"
DISPERSED_DEMAND_GWH_BY_CLUSTER = {1: 2.079, 2: 0.132, 3: 0.648, 4: 1.543, 5: 0.000}
EXCLUDE_TECHS_FOR_OFFRE_LOCALE = {"ELECTRICITY", "PV_HS", "HS_DIESEL", "BATT_HS"}
RECAL_SCENARIOS = ["late_access", "early_access"]

DEPLOYED_SHARE_DISPERSION_RECAL2 = {}
for scenario in RECAL_SCENARIOS:
    cs_outputs = f"{CS_ROOT_2050}/norte_amazonia_{scenario}_2050/outputs"

    solve_info = pd.read_csv(f"{cs_outputs}/Solve_info.csv", sep=r"\t;\t", header=None,
                              index_col=0, engine="python")
    solve_result_num = int(float(solve_info.loc["solve_result_num", 1]))
    assert solve_result_num == 0, (
        f"{scenario}_2050: solve_result_num={solve_result_num} != 0 -- "
        f"refusing to calibrate share_dispersion from a non-optimal solve")

    cs_dir = f"{cs_outputs}/regional_results"
    yb = pd.read_csv(f"{cs_dir}/Year_balance.csv", sep=";")
    yb["ELECTRICITY"] = pd.to_numeric(yb["ELECTRICITY"], errors="coerce")
    yb_tech = yb[~yb["Elements"].isin(EXCLUDE_TECHS_FOR_OFFRE_LOCALE)]
    yb_tech = yb_tech[yb_tech["ELECTRICITY"] > 1e-9]
    tech_sum = yb_tech.groupby("Regions")["ELECTRICITY"].sum()

    res = pd.read_csv(f"{cs_dir}/Resources.csv", sep=";")
    res_elec = res[res["Resources"] == "ELECTRICITY"].set_index("Regions")
    r_local = pd.to_numeric(res_elec["R_year_local"], errors="coerce").fillna(0.0)
    r_ext = pd.to_numeric(res_elec["R_year_exterior"], errors="coerce").fillna(0.0)

    shares, offres = {}, {}
    for k in range(1, 6):
        region = f"C{k}"
        offre_locale = (float(tech_sum.get(region, 0.0))
                         + float(r_local.get(region, 0.0)) + float(r_ext.get(region, 0.0)))
        target = DISPERSED_DEMAND_GWH_BY_CLUSTER[k]
        share = target / (target + offre_locale) if (target + offre_locale) > 0 else 0.0
        shares[k] = share
        offres[k] = offre_locale

    print(f"{scenario}_2050 (solve_result_num={solve_result_num}):")
    print(f"  offre_locale_hors_TECH_HS = {offres}")
    print(f"  share_dispersion (raw)    = {shares}")

    deployed = dict(shares)
    deployed[2] = 0.0
    deployed[5] = 0.0
    DEPLOYED_SHARE_DISPERSION_RECAL2[scenario] = deployed
    print(f"  deployed (C2/C5 forced 0) = {deployed}")
    print()

print("=== Comparison against the prior (now stale) calibration ===")
for scenario in RECAL_SCENARIOS:
    for k in range(1, 6):
        old = json.load(open(f"{ROOT}/Data/2050/{scenario}/C{k}/Misc.json"))["share_dispersion"]
        new = DEPLOYED_SHARE_DISPERSION_RECAL2[scenario][k]
        print(f"{scenario} C{k}: old={old:.6f}  new={new:.6f}")


late_access_2050 (solve_result_num=0):
  offre_locale_hors_TECH_HS = {1: 36.59150214913401, 2: 4.022596646145117, 3: 196.5372530360024, 4: 93.40178866724722, 5: 102.04286882163109}
  share_dispersion (raw)    = {1: 0.05376190854678512, 2: 0.03177203739440687, 3: 0.003286249808355025, 4: 0.016251550207855518, 5: 0.0}
  deployed (C2/C5 forced 0) = {1: 0.05376190854678512, 2: 0.0, 3: 0.003286249808355025, 4: 0.016251550207855518, 5: 0.0}

early_access_2050 (solve_result_num=0):
  offre_locale_hors_TECH_HS = {1: 36.59147474458595, 2: 4.0273046519175, 3: 193.8015788663832, 4: 96.6309232791451, 5: 101.67839519301495}
  share_dispersion (raw)    = {1: 0.053761946646157216, 2: 0.031736073946674256, 3: 0.0033324834323517653, 4: 0.0157170045615135, 5: 0.0}
  deployed (C2/C5 forced 0) = {1: 0.053761946646157216, 2: 0.0, 3: 0.0033324834323517653, 4: 0.0157170045615135, 5: 0.0}

=== Comparison against the prior (now stale) calibration ===
late_access C1: old=0.050638  new=0.053762
late_access C2: o

## Deploy to `Misc.json`, with assertion (late_access/early_access only)

In [2]:
for scenario in RECAL_SCENARIOS:
    for k in range(1, 6):
        target = f"{ROOT}/Data/2050/{scenario}/C{k}/Misc.json"
        deployed_value = DEPLOYED_SHARE_DISPERSION_RECAL2[scenario][k]

        with open(target, encoding="utf-8") as f:
            misc = json.load(f)
        misc["share_dispersion"] = deployed_value
        with open(target, "w", encoding="utf-8") as f:
            json.dump(misc, f, indent=4)

        with open(target, encoding="utf-8") as f:
            check = json.load(f)
        actual = check["share_dispersion"]
        if abs(actual - deployed_value) > 1e-9:
            raise AssertionError(f"{target}: share_dispersion = {actual}, expected {deployed_value}")
    print(f"OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for {scenario}_2050")


OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for late_access_2050
OK -- share_dispersion deployed and verified in Misc.json (C1-C5) for early_access_2050


## Reprint `reg_misc.dat` from the deployed catalog (no solve)

In [3]:
import sys
sys.path.insert(0, r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
from pathlib import Path
from esmc import Esmc

REPRINT_YEAR = 2050
FT_TO_DROP = ['BIOMASS_TO_GASOLINE', 'BIOMASS_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_DIESEL',
              'POWER_TO_GASOLINE', 'POWER_TO_DIESEL', 'H2_TO_GASOLINE', 'H2_TO_DIESEL']
AMPL_PATH = r'C:\Users\valen\AMPL'

REPRINT_MODELS = {}
for scenario in RECAL_SCENARIOS:
    case_study = f"norte_amazonia_{scenario}_{REPRINT_YEAR}"
    config = {'case_study': case_study, 'comment': 'share_dispersion recal #2 reprint (no solve)',
              'regions_names': ['C1', 'C2', 'C3', 'C4', 'C5'],
              'gwp_limit_overall': None, 're_share_primary': None, 'f_perc': True,
              'year': REPRINT_YEAR, 'scenario': scenario}

    my_model = Esmc(config, nbr_td=16)
    current_project = Path(r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
    my_model.project_dir = current_project
    my_model.dat_dir = current_project / 'case_studies' / my_model.space_id / '00_td_dat'
    my_model.cs_dir = current_project / 'case_studies' / my_model.space_id / case_study
    my_model.dat_dir.mkdir(parents=True, exist_ok=True)
    my_model.cs_dir.mkdir(parents=True, exist_ok=True)

    my_model.read_data_indep()
    my_model.init_regions()

    my_model.ref_region.data['Technologies'] = my_model.ref_region.data['Technologies'].drop(index=FT_TO_DROP)
    my_model.data_indep['Layers_in_out'] = my_model.data_indep['Layers_in_out'].drop(index=FT_TO_DROP)
    for r_code, region in my_model.regions.items():
        region.data['Technologies'] = region.data['Technologies'].drop(index=FT_TO_DROP)

    assert case_study.startswith('norte_amazonia_early_access_') or case_study.startswith('norte_amazonia_late_access_')
    for r_code, region in my_model.regions.items():
        region.data['Technologies'].loc['PV_UTILITY', 'f_max'] = 1e15
        region.data['Technologies'].loc['BATT_LI', 'f_max'] = 1e15

    for k in range(1, 6):
        region_code = f"C{k}"
        in_memory_sd = float(my_model.regions[region_code].data['Misc']['share_dispersion'])
        expected = DEPLOYED_SHARE_DISPERSION_RECAL2[scenario][k]
        assert abs(in_memory_sd - expected) < 1e-9, (
            f"{scenario} {region_code}: in-memory share_dispersion={in_memory_sd} after "
            f"init_regions(), expected {expected} (deployed Misc.json)")

    my_model.init_ta(algo='read', ampl_path=AMPL_PATH)
    my_model.print_td_data()
    my_model.print_data(indep=True)

    REPRINT_MODELS[scenario] = my_model
    print(f"OK -- reg_misc.dat reprinted for {case_study} at {my_model.cs_dir} "
          f"(pre-print in-memory check passed, no solve)")

    dat_path = my_model.cs_dir / "reg_misc.dat"
    lines = dat_path.read_text(encoding="utf-8").splitlines()
    header_line_idx = next(i for i, l in enumerate(lines)
                            if l.strip().startswith("param") and "share_dispersion" in l)
    header_fields = lines[header_line_idx].split()
    col_idx = header_fields.index("share_dispersion") - 1
    for k in range(1, 6):
        row = next((l for l in lines[header_line_idx+1:] if l.split() and l.split()[0] == f"C{k}"), None)
        assert row is not None, f"reg_misc.dat: no C{k} row found in the share_dispersion block"
        printed_sd = float(row.split()[col_idx])
        expected = DEPLOYED_SHARE_DISPERSION_RECAL2[scenario][k]
        assert abs(printed_sd - expected) < 1e-6, (
            f"reg_misc.dat C{k}: printed share_dispersion={printed_sd}, expected {expected}")
    print(f"OK -- reg_misc.dat text spot-check passed for {scenario} (all 5 clusters)")


[INFO    ] (read_data_indep): Read indep data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\late_access\00_INDEP


[INFO    ] (init_regions): Initialising regions: C1, C2, C3, C4, C5


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\late_access\02_REF_REGION


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\late_access\C1


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\late_access\C2


...

[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_td_data): Printing TD data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_2050\reg_16TD.dat


[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_data): Printing regional data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_2050


OK -- reg_misc.dat reprinted for norte_amazonia_early_access_2050 at C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_2050 (pre-print in-memory check passed, no solve)
OK -- reg_misc.dat text spot-check passed for early_access (all 5 clusters)
